# Part 1: Data Assessment

## Objective

Examine existing arXiv research papers in the Artificial Intelligence category, identify the available file types and metadata,
assess the quality of the data, and determine what information is usable for the RAG pipeline.

In [21]:
# setup, imports

import io
import time
import requests
import feedparser
from pathlib import Path
import pandas as pd
from pypdf import PdfReader

cwd = Path.cwd()
project_root = cwd.parent if cwd.name == "notebooks" else cwd

In [23]:
# define arxiv fetch + extraction functions

ARXIV_API_URL = "http://export.arxiv.org/api/query"

def sanitize_text(text):
    return text.encode("utf-8", errors="ignore").decode("utf-8") if isinstance(text, str) else text

def fetch_arxiv_metadata(query, total_results, batch_size=100):
    papers = []
    for start in range(0, total_results, batch_size):
        params = {
            "search_query": query,
            "start": start,
            "max_results": min(batch_size, total_results - start),
            "sortBy": "submittedDate",
            "sortOrder": "descending",
        }
        response = requests.get(ARXIV_API_URL, params=params, timeout=30)
        response.raise_for_status()
        feed = feedparser.parse(response.text)

        for entry in feed.entries:
            arxiv_id = entry.id.split("/abs/")[-1]
            pdf_url = next(
                (link.href for link in entry.links if link.get("title") == "pdf"), None
            )
            papers.append({
                "arxiv_id": arxiv_id,
                "title": entry.title.replace("\n", " ").strip(),
                "author": "; ".join(a.name for a in entry.authors),
                "published": entry.published,
                "pdf_url": pdf_url,
            })

        print(f"fetched metadata for {len(papers)}/{total_results} papers")
        time.sleep(3)
    return papers


def extract_pdf_pages_from_url(paper):
    response = requests.get(paper["pdf_url"], timeout=30)
    response.raise_for_status()
    reader = PdfReader(io.BytesIO(response.content))

    records = []
    for page_number, page in enumerate(reader.pages, start=1):
        raw_text = sanitize_text(page.extract_text() or "")
        records.append({
            "arxiv_id": paper["arxiv_id"],
            "title": sanitize_text(paper["title"]),
            "author": sanitize_text(paper["author"]),
            "published": paper["published"],
            "page_number": page_number,
            "num_pages": len(reader.pages),
            "raw_text": raw_text,
        })
    return records

In [6]:
# fetch metadata and extract pages

# skip this cell if you're about to run the ArxivIngestionPipeline class (in Part 2) below instead, since running both fetches everything twice

paper_metadata = fetch_arxiv_metadata("cat:cs.AI", total_results=500) # importing 500 papers for now for testing the api, but when we scale to ingesting updated+new papers every day, we'd have to export to a .py script and run a scheduled workflow for daily imports
metadata_df = pd.DataFrame(paper_metadata)

all_pages = []
for i, paper in enumerate(paper_metadata, start=1):
    if paper["pdf_url"] is None:
        continue
    try:
        all_pages.extend(extract_pdf_pages_from_url(paper))
    except Exception as e:
        print(f"failed to extract {paper['arxiv_id']}: {e}")
    if i % 25 == 0:
        print(f"processed {i}/{len(paper_metadata)} papers")
    time.sleep(3) # arxiv rate limit policy asks for no more than 1 request every 3 seconds

pages_df = pd.DataFrame(all_pages)
print(f"extracted {len(pages_df)} pages from {pages_df['arxiv_id'].nunique()} papers")

fetched metadata for 100/500 papers
fetched metadata for 200/500 papers
fetched metadata for 300/500 papers
fetched metadata for 400/500 papers
fetched metadata for 500/500 papers
processed 25/500 papers
processed 50/500 papers


Exceeded 5000 form XObject invocations while extracting text; further form content is skipped.


processed 75/500 papers
processed 100/500 papers


Exceeded 5000 form XObject invocations while extracting text; further form content is skipped.
Exceeded 5000 form XObject invocations while extracting text; further form content is skipped.
Exceeded 5000 form XObject invocations while extracting text; further form content is skipped.
Exceeded 5000 form XObject invocations while extracting text; further form content is skipped.


processed 125/500 papers


Exceeded 5000 form XObject invocations while extracting text; further form content is skipped.
Impossible to decode XFormObject /Im13: Reached maximum number of tokens per stream.


... (13 outputs left out)

could not convert string to float: b'0.000000000000-5684342' : FloatObject (b'0.000000000000-5684342') invalid; use 0.0 instead
could not convert string to float: b'0.000000000000-5684342' : FloatObject (b'0.000000000000-5684342') invalid; use 0.0 instead
could not convert string to float: b'0.000000000000-5684342' : FloatObject (b'0.000000000000-5684342') invalid; use 0.0 instead
Skipping broken line b'14f6   14ff   10300': Odd-length string
Skipping broken line b'1500   1514   1030a': Odd-length string
Skipping broken line b'1515   1518   10320': Odd-length string
Skipping broken line b'1519   156f   1d300': Odd-length string
Skipping broken line b'1570   1571   1d538': Odd-length string
Skipping broken line b'1572   1575   1d53b': Odd-length string
Skipping broken line b'1576   157a   1d540': Odd-length string
Skipping broken line b'157c   1582   1d54a': Odd-length string
Skipping broken line b'1583   159c   1d552': Odd-length string
Skipping broken line b'159d   15d0   1d5d4': Odd-

processed 475/500 papers


Skipping broken line b'11cd   11d2   10780': Odd-length string
Skipping broken line b'11d3   11fc   10787': Odd-length string
Skipping broken line b'11fd   11ff   107b2': Odd-length string
Skipping broken line b'1200   1205   107b5': Odd-length string
Skipping broken line b'1206   1224   1df00': Odd-length string
Skipping broken line b'1225   122a   1df25': Odd-length string
Skipping broken line b'0d04   0d09   10780': Odd-length string
Skipping broken line b'0d0a   0d33   10787': Odd-length string
Skipping broken line b'0d34   0d3c   107b2': Odd-length string
Skipping broken line b'0d3d   0d5b   1df00': Odd-length string
Skipping broken line b'0d5c   0d61   1df25': Odd-length string
Skipping broken line b'11cd   11d2   10780': Odd-length string
Skipping broken line b'11d3   11fc   10787': Odd-length string
Skipping broken line b'11fd   11ff   107b2': Odd-length string
Skipping broken line b'1200   1205   107b5': Odd-length string
Skipping broken line b'1206   1224   1df00': Odd-length

processed 500/500 papers
extracted 12133 pages from 500 papers


In [7]:
# aggregate quality checks (depends on fetching cell above, also skip if running ArxivIngestionPipeline below)

paper_quality = pages_df.groupby("arxiv_id").agg(
    num_pages=("page_number", "max"),
    pages_with_text=("raw_text", lambda s: (s.str.strip() != "").sum()),
).reset_index()
paper_quality["readable"] = paper_quality["pages_with_text"] > 0

print("papers fetched:", pages_df["arxiv_id"].nunique())
print("unreadable papers:", (~paper_quality["readable"]).sum())
print("duplicate arxiv_ids:", metadata_df["arxiv_id"].duplicated().sum())
print("missing titles:", metadata_df["title"].isna().sum())
print("missing pdf links:", metadata_df["pdf_url"].isna().sum())

paper_quality[~paper_quality["readable"]]

papers fetched: 500
unreadable papers: 0
duplicate arxiv_ids: 0
missing titles: 0
missing pdf links: 0


,arxiv_id,num_pages,pages_with_text,readable


In [8]:
# spot check extracted text

sample_papers = pages_df[pages_df["page_number"] == 1].head(5)

for _, row in sample_papers.iterrows():
    print("\nARXIV ID:", row["arxiv_id"])
    print("TITLE:", row["title"])
    print(row["raw_text"][:300])
    print("-" * 80)


ARXIV ID: 2610.35868v1
TITLE: Retriever-Free Retrieval: Navigating Documents with Decision Models
Retriever-Free Retrieval: Navigating Documents with Decision Models
Chao Wan * 1 Paul J¨unger * 1 Beixi Du 1 Bodong Liu 1 Navaneetha Krishnan 2 Kilian Q. Weinberger1 2
Abstract
While language models have become dramatically
more capable, retrieval still largely relies on lex-
ical matching or simila
--------------------------------------------------------------------------------

ARXIV ID: 2610.35865v1
TITLE: Center-of-Mass Interpretation of Evolution Strategies
Center-of-Mass Interpretation of Evolution Strategies
Akshat Gupta
UC Berkeley
akshat.gupta@berkeley.edu
Abstract
Evolution strategies (ES) update a model by
sampling a population of perturbed copies,
scoring each one, and moving in the reward-
weighted direction of the perturbations. This
update is
--------------------------------------------------------------------------------

ARXIV ID: 2610.35862v1
TITLE: Learning Force-Reacti

## Data Quality Findings

- Imported 500 most recent arXiv research papers, all in PDF format.
- All 500 PDFs opened successfully and contained extractable text on every page.
- Titles and author information were available for all papers.
- No duplicate filenames or paper titles were found.
- No missing values were found in the fields collected during the assessment.



# Part 2: Data Prep

In [24]:
# extra imports, output dir

import re
import json
import unicodedata
from dataclasses import dataclass
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("ingestion_pipeline")

processed_dir = project_root / "data_processed"
processed_dir.mkdir(exist_ok=True)

print("Processed output directory:", processed_dir)

Processed output directory: /Users/anna/KPMG-1K-ai-research-intelligence-agent-for-business-insight-translation/data_processed


In [26]:
# defining functions for cleaning and chunking text

def clean_text(text: str) -> str:
    """Clean raw PDF-extracted text based on artifacts identified in the Part 1 assessment."""
    if not text or not text.strip():
        return ""

    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"-\s*\n\s*", "", text)              # rejoin hyphenated line breaks
    text = re.sub(r"(\w)-\s+(\w)", r"\1-\2", text)      # rejoin hyphenated stray spacing
    text = re.sub(r"\s[\*†‡]\s", " ", text)             # strip isolated footnote markers
    text = re.sub(r"\n+", " ", text)                    # collapse newlines
    text = re.sub(r"[ \t]+", " ", text)                 # collapse repeated whitespace

    return text.strip()

CHUNK_SIZE_WORDS = 250
CHUNK_OVERLAP_WORDS = 40

def chunk_text(text: str, chunk_size: int = CHUNK_SIZE_WORDS, overlap: int = CHUNK_OVERLAP_WORDS) -> list[str]:
    words = text.split()
    if not words:
        return []
    if len(words) <= chunk_size:
        return [text]

    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunks.append(" ".join(words[start:end]))
        start += chunk_size - overlap
    return chunks

In [ ]:
# clean text

pages_df["cleaned_text"] = pages_df["raw_text"].apply(clean_text)
pages_df["char_count"] = pages_df["cleaned_text"].str.len()
pages_df["word_count"] = pages_df["cleaned_text"].str.split().str.len()

print("BEFORE:\n", pages_df.loc[0, "raw_text"][:300])
print("\nAFTER:\n", pages_df.loc[0, "cleaned_text"][:300])

BEFORE:
 Retriever-Free Retrieval: Navigating Documents with Decision Models
Chao Wan * 1 Paul J¨unger * 1 Beixi Du 1 Bodong Liu 1 Navaneetha Krishnan 2 Kilian Q. Weinberger1 2
Abstract
While language models have become dramatically
more capable, retrieval still largely relies on lex-
ical matching or simila

AFTER:
 Retriever-Free Retrieval: Navigating Documents with Decision Models Chao Wan 1 Paul J ̈unger 1 Beixi Du 1 Bodong Liu 1 Navaneetha Krishnan 2 Kilian Q. Weinberger1 2 Abstract While language models have become dramatically more capable, retrieval still largely relies on lexical matching or similarity 


In [11]:
# chunk text

chunk_records = []
for _, row in pages_df.iterrows():
    page_chunks = chunk_text(row["cleaned_text"])
    for chunk_index, chunk in enumerate(page_chunks):
        chunk_records.append({
            "chunk_id": f"{row['arxiv_id']}_p{row['page_number']}_c{chunk_index}",
            "arxiv_id": row["arxiv_id"],
            "title": row["title"],
            "author": row["author"],
            "published": row["published"],
            "page_number": row["page_number"],
            "chunk_index": chunk_index,
            "text": chunk,
            "word_count": len(chunk.split())
        })

chunks_df = pd.DataFrame(chunk_records)
print(f"Produced {len(chunks_df)} chunks from {pages_df['arxiv_id'].nunique()} papers")
chunks_df[["chunk_id", "page_number", "word_count"]].head(10)

Produced 35356 chunks from 500 papers


,chunk_id,page_number,word_count
0,2610.35868v1_p1_c0,1,250
1,2610.35868v1_p1_c1,1,250
2,2610.35868v1_p1_c2,1,210
3,2610.35868v1_p2_c0,2,250
4,2610.35868v1_p2_c1,2,250
5,2610.35868v1_p2_c2,2,250
6,2610.35868v1_p2_c3,2,144
7,2610.35868v1_p3_c0,3,250
8,2610.35868v1_p3_c1,3,250
9,2610.35868v1_p3_c2,3,250


In [12]:
# validate

assert not chunks_df.empty, "No chunks were produced"
assert chunks_df["chunk_id"].is_unique, "Duplicate chunk IDs found"
assert (chunks_df["word_count"] > 0).all(), "Empty chunks found"
assert chunks_df["title"].notna().all(), "Chunks missing title metadata"
assert chunks_df["arxiv_id"].notna().all(), "Chunks missing arxiv_id"
assert (chunks_df["word_count"] <= CHUNK_SIZE_WORDS + 5).all(), "Chunk exceeds target size unexpectedly"

print("All validation checks passed.")
print("\nChunk word-count distribution:")
print(chunks_df["word_count"].describe())

All validation checks passed.

Chunk word-count distribution:
count    35356.000000
mean       201.123883
std         75.099395
min          1.000000
25%        160.000000
50%        250.000000
75%        250.000000
max        250.000000
Name: word_count, dtype: float64


In [13]:
#save

jsonl_path = processed_dir / "chunks.jsonl"
parquet_path = processed_dir / "chunks.parquet"

with open(jsonl_path, "w") as f:
    for record in chunks_df.to_dict(orient="records"):
        f.write(json.dumps(record) + "\n")

chunks_df.to_parquet(parquet_path, index=False)

logger.info(f"Saved {len(chunks_df)} chunks to {jsonl_path}")
logger.info(f"Saved {len(chunks_df)} chunks to {parquet_path}")

2026-10-27 01:14:17,167 | INFO | Saved 35356 chunks to /Users/eimansherzada/Desktop/BTT-KPMG/KPMG-1K-ai-research-intelligence-agent-for-business-insight-translation/data_processed/chunks.jsonl
2026-10-27 01:14:17,168 | INFO | Saved 35356 chunks to /Users/eimansherzada/Desktop/BTT-KPMG/KPMG-1K-ai-research-intelligence-agent-for-business-insight-translation/data_processed/chunks.parquet


In [27]:
# wrap as a repeatable pipeline 

@dataclass
class IngestionConfig:
    query: str
    total_results: int
    output_dir: Path
    chunk_size_words: int = CHUNK_SIZE_WORDS
    chunk_overlap_words: int = CHUNK_OVERLAP_WORDS


class ArxivIngestionPipeline:
    def __init__(self, config: IngestionConfig):
        self.config = config
        self.pages_df = None
        self.chunks_df = None

    def extract(self):
        metadata = fetch_arxiv_metadata(self.config.query, self.config.total_results)
        records = []
        for i, paper in enumerate(metadata, start=1):
            if paper["pdf_url"] is None:
                continue
            try:
                records.extend(extract_pdf_pages_from_url(paper))
            except Exception as e:
                logger.warning(f"Failed to extract {paper['arxiv_id']}: {e}")
            if i % 25 == 0:
                logger.info(f"Processed {i}/{len(metadata)} papers")
            time.sleep(3)
        self.pages_df = pd.DataFrame(records)
        return self

    def clean(self):
        self.pages_df["cleaned_text"] = self.pages_df["raw_text"].apply(clean_text)
        return self

    def chunk(self):
        records = []
        for _, row in self.pages_df.iterrows():
            page_chunks = chunk_text(
                row["cleaned_text"],
                chunk_size=self.config.chunk_size_words,
                overlap=self.config.chunk_overlap_words
            )
            for chunk_index, chunk in enumerate(page_chunks):
                records.append({
                    "chunk_id": f"{row['arxiv_id']}_p{row['page_number']}_c{chunk_index}",
                    "arxiv_id": row["arxiv_id"],
                    "title": row["title"],
                    "author": row["author"],
                    "published": row["published"],
                    "page_number": row["page_number"],
                    "chunk_index": chunk_index,
                    "text": chunk,
                    "word_count": len(chunk.split())
                })
        self.chunks_df = pd.DataFrame(records)
        return self

    def validate(self):
        assert not self.chunks_df.empty, "No chunks produced"
        assert self.chunks_df["chunk_id"].is_unique, "Duplicate chunk IDs"
        assert (self.chunks_df["word_count"] > 0).all(), "Empty chunks found"
        assert self.chunks_df["title"].notna().all(), "Missing title metadata"
        return self

    def load(self):
        self.config.output_dir.mkdir(exist_ok=True)
        jsonl_path = self.config.output_dir / "chunks.jsonl"
        with open(jsonl_path, "w") as f:
            for record in self.chunks_df.to_dict(orient="records"):
                f.write(json.dumps(record) + "\n")
        self.chunks_df.to_parquet(self.config.output_dir / "chunks.parquet", index=False)
        logger.info(f"Saved {len(self.chunks_df)} chunks to {self.config.output_dir}")
        return self

    def run(self):
        return self.extract().clean().chunk().validate().load()

In [28]:
# only run this if you haven't already run the manual walkthrough above in this session, since running both fetches everything twice

config = IngestionConfig(query="cat:cs.AI", total_results=500, output_dir=processed_dir)
pipeline = ArxivIngestionPipeline(config)
pipeline.run()
print(f"Pipeline complete: {len(pipeline.chunks_df)} chunks ready for embedding")

fetched metadata for 100/500 papers
fetched metadata for 200/500 papers
fetched metadata for 300/500 papers
fetched metadata for 400/500 papers
fetched metadata for 500/500 papers


2026-11-08 11:53:10,831 | WARNING | Skipping broken line b'14f9   14ff   10300': Odd-length string
2026-11-08 11:53:10,831 | WARNING | Skipping broken line b'1500   1517   10307': Odd-length string
2026-11-08 11:53:10,831 | WARNING | Skipping broken line b'1518   151b   10320': Odd-length string
2026-11-08 11:53:10,831 | WARNING | Skipping broken line b'151c   1572   1d300': Odd-length string
2026-11-08 11:53:10,832 | WARNING | Skipping broken line b'1573   1574   1d538': Odd-length string
2026-11-08 11:53:10,832 | WARNING | Skipping broken line b'1575   1578   1d53b': Odd-length string
2026-11-08 11:53:10,832 | WARNING | Skipping broken line b'1579   157d   1d540': Odd-length string
2026-11-08 11:53:10,832 | WARNING | Skipping broken line b'157f   1585   1d54a': Odd-length string
2026-11-08 11:53:10,832 | WARNING | Skipping broken line b'1586   159f   1d552': Odd-length string
2026-11-08 11:53:10,832 | WARNING | Skipping broken line b'15a0   15d3   1d5d4': Odd-length string
2026-11-08

Pipeline complete: 38008 chunks ready for embedding


## Summary

- Extracted per-page text and metadata for all 500 papers.
- Cleaned hyphenation breaks, stray footnote symbols, and irregular whitespace found during Part 1.
- Chunked text at the page level (250 words, 40-word overlap) so every chunk keeps an accurate page number for citations.
- Validated that every chunk has complete metadata, a unique ID, and non-empty text.
- Packaged the flow into `ArxivIngestionPipeline` for repeatable reruns.
- Output: `data_processed/chunks.jsonl` and `chunks.parquet`, ready for embedding.

# Part 3: RAG Model Build

In [29]:
# install dependencies
# llamaindex is the core framework that wraps the other components (embeddings, vector stores, LLMs) together
# chromadb is a vector database for the embeddings

%pip install llama-index-core llama-index-embeddings-huggingface llama-index-vector-stores-chroma llama-index-llms-groq chromadb

import os
import chromadb

from llama_index.core import VectorStoreIndex, StorageContext, Settings
from llama_index.core.schema import TextNode
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core.postprocessor import SentenceTransformerRerank
from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.core.retrievers import QueryFusionRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.llms.groq import Groq


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [30]:
# load chunks

chunks_df = pd.read_parquet(processed_dir / "chunks.parquet")
print(f"loaded {len(chunks_df)} chunks from {processed_dir / 'chunks.parquet'}")

loaded 38008 chunks from /Users/anna/KPMG-1K-ai-research-intelligence-agent-for-business-insight-translation/data_processed/chunks.parquet


In [31]:
# convert raw chunks into llamaindex textnodes, i.e. units with text and metadata

nodes = []
for _, row in chunks_df.iterrows():
    node = TextNode(
        text=row["text"],
        metadata={
            "arxiv_id": row["arxiv_id"],
            "title": row["title"],
            "author": row["author"],
            "page_number": row["page_number"],
        }
    )
    nodes.append(node)

print(f"built {len(nodes)} nodes")

built 38008 nodes


In [32]:
# config embedding model

Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

2026-11-08 12:28:30,688 | INFO | HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-11-08 12:28:30,689 | WARNING | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-11-08 12:28:30,790 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/modules.json?%2FBAAI%2Fbge-small-en-v1.5%2Fresolve%2Fmain%2Fmodules.json=&etag=%22952a9b81c0bfd99800fabf352f69c7ccd46c5e43%22 "HTTP/1.1 200 OK"
2026-11-08 12:28:30,817 | INFO | HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-11-08 12:28:30,832 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config_sentence_t

In [33]:
# config llm used to generate responses

from dotenv import load_dotenv
load_dotenv()  # load groq api key from .env file

Settings.llm = Groq(model="openai/gpt-oss-120b") # use openai/gpt-oss-120b model instead if hit rate limit, but could have lower answer quality

In [34]:
# ADDED: explicitly reload the Groq API key from the project .env file
# to fix the 401 authentication error caused by a stale or missing key.

from dotenv import load_dotenv
import os
from llama_index.llms.groq import Groq

load_dotenv(project_root / ".env", override=True)

groq_api_key = os.getenv("GROQ_API_KEY")

print("Groq key loaded:", bool(groq_api_key))

Settings.llm = Groq(
    model="openai/gpt-oss-120b",
    api_key=groq_api_key
)

Groq key loaded: True


In [35]:
# set up chroma for vectore store and build index

chroma_client = chromadb.PersistentClient(path=str(processed_dir / "chroma_db"))
chroma_collection = chroma_client.get_or_create_collection(name="arxiv_papers")

vectore_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vectore_store)

index = VectorStoreIndex(nodes, storage_context=storage_context)

print("index built and persisted to", processed_dir / "chroma_db")

index built and persisted to /Users/anna/KPMG-1K-ai-research-intelligence-agent-for-business-insight-translation/data_processed/chroma_db


In [36]:
# hybrid retrieval: bm25 (keyword) + vector (semantic), fused via reciprocal rank fusion

bm25_retriever = BM25Retriever.from_defaults(nodes=nodes, similarity_top_k=30)
vector_retriever = index.as_retriever(similarity_top_k=30)

fusion_retriever = QueryFusionRetriever(
    [vector_retriever, bm25_retriever],
    similarity_top_k=30,
    num_queries=1,
    mode="reciprocal_rerank",
)

2026-11-08 15:10:28,918 | DEBUG | Building index from IDs objects


In [37]:
# build query engine (embed question, retrieve top similar chunks, feed them into llm, and generate response)

reranker = SentenceTransformerRerank(model="cross-encoder/ms-marco-MiniLM-L-6-v2", top_n=10)
query_engine = RetrieverQueryEngine.from_args(
    retriever=fusion_retriever,
    node_postprocessors=[reranker]
)

2026-11-08 15:10:33,094 | INFO | HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L-6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-11-08 15:10:33,136 | INFO | HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 404 Not Found"
2026-11-08 15:10:33,140 | INFO | No modules.json found for cross-encoder/ms-marco-MiniLM-L-6-v2, initializing a new CrossEncoder model.
2026-11-08 15:10:33,241 | INFO | HTTP Request: GET https://huggingface.co/api/models/cross-encoder/ms-marco-MiniLM-L-6-v2 "HTTP/1.1 307 Temporary Redirect"
2026-11-08 15:10:33,296 | INFO | HTTP Request: GET https://huggingface.co/api/models/cross-encoder/ms-marco-MiniLM-L6-v2 "HTTP/1.1 200 OK"
2026-11-08 15:10:33,338 | INFO | HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L-6-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-11-08 15:10:33,370 | INFO | HTTP Request: HEAD https://hugg

In [38]:
# ask question (hardcoded for now)

response = query_engine.query("What are the most relevant recent research papers on reducing hallucinations in retrieval-augmented generation? Summarize the main approaches and identify where the evidence is strongest.")

print("RESPONSE:\n", response)

print("\nSOURCES:")
for node in response.source_nodes:
    meta = node.metadata
    print(f"- {meta['title']} (page {meta['page_number']}, score={node.score:.3f})")

Batches: 100%|██████████| 1/1 [00:00<00:00,  7.62it/s]
2026-11-08 15:10:48,605 | INFO | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-08 15:10:51,414 | INFO | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


RESPONSE:
 **Key recent papers that focus on curbing hallucinations in retrieval‑augmented generation (RAG)**  

| Paper | Year | Core Idea for Hallucination Mitigation |
|-------|------|----------------------------------------|
| *When Not to Trust Language Models* (Mallen et al.) | 2023 | Systematic analysis of when parametric knowledge fails and how non‑parametric (retrieved) memory can be used to flag unreliable outputs. |
| *Redeep: Detecting Hallucination in Retrieval‑Augmented Generation via Mechanistic Interpretability* (Sun et al.) | 2025 | Uses mechanistic interpretability to expose the internal pathways that lead to hallucinated tokens, enabling a detection layer that can reject or correct them. |
| *Halueval: A Large‑Scale Hallucination Evaluation Benchmark for Large Language Models* (Li et al.) | 2023 | Provides a comprehensive benchmark suite that quantifies hallucination rates across diverse tasks, giving a solid empirical basis for comparing mitigation techniques. |
| *

## Part 3B: End-to-End Retrieval + Summarization Workflow

This section tests the complete retrieval and summarization workflow across the expanded arXiv dataset. The workflow accepts a natural-language research question, retrieves and reranks relevant research chunks, generates a summarized response with the configured LLM, and returns source metadata for review.

In [39]:
#reusable end-to-end retrieval + summarization function
# Takes a research question, runs it through the existing RAG query engine,
# and returns both the generated answer and source metadata.

def run_research_query(question, top_k=5):
    """
    Run a research question through the complete RAG pipeline
    and return the generated response with retrieved source metadata.
    """

    response = query_engine.query(question)

    sources = []

    for node in response.source_nodes[:top_k]:
        sources.append({
            "title": node.metadata.get("title"),
            "arxiv_id": node.metadata.get("arxiv_id"),
            "page_number": node.metadata.get("page_number"),
            "score": node.score
        })

    return {
        "question": question,
        "answer": str(response),
        "sources": sources
    }

In [40]:
#  test the end-to-end workflow using research questions
# covering several AI topics in the expanded arXiv dataset.

test_questions = [
    "What recent methods are being used to reduce hallucinations in retrieval-augmented generation?",
    "What approaches are recent AI research papers using to improve AI agent reliability?",
    "What techniques are being explored to make large language models more efficient?",
    "What recent research focuses on improving the safety or trustworthiness of generative AI?",
    "How are researchers improving retrieval and reasoning in AI systems?"
]

In [41]:
# Run each research question through the complete workflow.

test_results = []

for question in test_questions:
    result = run_research_query(question)
    test_results.append(result)

    print("=" * 80)
    print("QUESTION:")
    print(result["question"])

    print("\nANSWER:")
    print(result["answer"])

    print("\nSOURCES:")
    for source in result["sources"]:
        print(
            f"- {source['title']} "
            f"(arXiv: {source['arxiv_id']}, "
            f"page {source['page_number']}, "
            f"score={source['score']:.3f})"
        )

    print()

Batches: 100%|██████████| 1/1 [00:00<00:00, 45.49it/s]
2026-11-08 15:11:20,494 | INFO | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-08 15:11:20,547 | INFO | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-11-08 15:11:20,551 | INFO | Retrying request to /chat/completions in 13.000000 seconds
2026-11-08 15:11:33,711 | INFO | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-11-08 15:11:33,715 | INFO | Retrying request to /chat/completions in 8.000000 seconds
2026-11-08 15:11:42,597 | INFO | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


QUESTION:
What recent methods are being used to reduce hallucinations in retrieval-augmented generation?

ANSWER:
Recent approaches focus on tightly coupling external knowledge with the generation process. By first retrieving relevant passages from a vector‑indexed store and feeding those passages into the language model, the system grounds its answers in factual material rather than relying solely on parametric knowledge. Enhancements that embed graph‑structured information—such as type‑ or role‑encoded word embeddings, graph neural or attention networks, and multi‑modal attention over knowledge‑graph features—further anchor the model’s reasoning to explicit relations in the source data. Decentralized retrieval frameworks that route queries to peers holding the needed documents (e.g., adaptive semantic peer routing) also improve the relevance and reliability of the retrieved context. Together, these retrieval‑first pipelines and graph‑grounded representations constitute the current to

Batches: 100%|██████████| 1/1 [00:00<00:00, 46.90it/s]
2026-11-08 15:11:43,667 | INFO | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-11-08 15:11:43,668 | INFO | Retrying request to /chat/completions in 17.000000 seconds
2026-11-08 15:12:03,041 | INFO | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-08 15:12:03,099 | INFO | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-11-08 15:12:03,099 | INFO | Retrying request to /chat/completions in 14.000000 seconds
2026-11-08 15:12:18,858 | INFO | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


QUESTION:
What approaches are recent AI research papers using to improve AI agent reliability?

ANSWER:
Recent AI research papers are converging on a multi‑layered toolbox for boosting the reliability of autonomous agents:

1. **Closing the interpretability gap** – By probing the internal circuitry of large models, researchers aim to understand how decisions are computed. This mechanistic insight enables the design of targeted control hooks and early‑warning signals for hidden failure modes, narrowing the disparity between what models can do and what we know about how they do it.

2. **Safety‑oriented reinforcement learning from human feedback (RLHF)** – Fine‑tuning agents with human‑generated preference data steers them toward behavior that aligns with human values. Modern RLHF pipelines incorporate iterative preference modeling, reward‑model calibration, and robustness checks to guard against deceptive alignment.

3. **Adversarial red‑team testing** – Systematic, optimization‑driven 

Batches: 100%|██████████| 1/1 [00:00<00:00, 49.68it/s]
2026-11-08 15:12:19,468 | INFO | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-11-08 15:12:19,469 | INFO | Retrying request to /chat/completions in 30.000000 seconds
2026-11-08 15:12:51,319 | INFO | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-08 15:12:51,377 | INFO | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-11-08 15:12:51,378 | INFO | Retrying request to /chat/completions in 13.000000 seconds
2026-11-08 15:13:04,469 | INFO | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-11-08 15:13:04,471 | INFO | Retrying request to /chat/completions in 2.000000 seconds
2026-11-08 15:13:07,952 | INFO | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


QUESTION:
What techniques are being explored to make large language models more efficient?

ANSWER:
A variety of complementary strategies are being investigated to cut the computational and energy costs of large language models while keeping their performance intact:

* **Query‑aware routing** – Incoming requests are classified on‑the‑fly; “easy” or latency‑critical queries are sent to smaller, faster model instances, while only the most demanding inputs are processed by the full‑size model. This dynamic allocation lets systems meet power‑ or latency‑budgets without sacrificing capability where it matters most.

* **Adaptive inference budgets** – At inference time the model can adjust how much reasoning it performs, for example by scaling the number of reasoning tokens, truncating generation steps, or aggregating several short responses. Computation is therefore expended only when higher accuracy is required.

* **Knowledge‑guided pruning and masking** – Parts of the network that contr

Batches: 100%|██████████| 1/1 [00:00<00:00, 32.04it/s]
2026-11-08 15:13:08,651 | INFO | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-11-08 15:13:08,652 | INFO | Retrying request to /chat/completions in 28.000000 seconds
2026-11-08 15:13:37,840 | INFO | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-08 15:13:37,958 | INFO | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-11-08 15:13:37,960 | INFO | Retrying request to /chat/completions in 13.000000 seconds
2026-11-08 15:13:51,096 | INFO | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-11-08 15:13:51,118 | INFO | Retrying request to /chat/completions in 2.000000 seconds
2026-11-08 15:13:54,707 | INFO | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


QUESTION:
What recent research focuses on improving the safety or trustworthiness of generative AI?

ANSWER:
Recent investigations have taken several complementary directions to make generative AI systems safer and more trustworthy.

* **Stakeholder‑oriented transparency** – Empirical surveys have mapped the distinct transparency requirements of end‑users, model builders, and policymakers.  The work shows that existing model cards, datasheets, and user‑facing disclosures often omit the provenance, development processes, and accountability mechanisms that these groups need to calibrate trust.  The authors propose participatory design of transparency tools, richer documentation standards, and a broader sociotechnical infrastructure (e.g., audit logs, provenance registries) to close those gaps.

* **Auditable unlearning** – A new framework enables large language models to prove that specific training data have been removed in response to deletion requests.  By coupling selective weight‑ma

Batches: 100%|██████████| 1/1 [00:00<00:00, 48.31it/s]
2026-11-08 15:13:55,331 | INFO | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-11-08 15:13:55,332 | INFO | Retrying request to /chat/completions in 26.000000 seconds
2026-11-08 15:14:23,170 | INFO | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-08 15:14:23,216 | INFO | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-11-08 15:14:23,217 | INFO | Retrying request to /chat/completions in 12.000000 seconds
2026-11-08 15:14:36,991 | INFO | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


QUESTION:
How are researchers improving retrieval and reasoning in AI systems?

ANSWER:
Researchers are tightening the loop between retrieval and reasoning by giving language models richer, more structured ways to decide what information to pull and when to use it.  Recent work equips LLM‑based agents with graph‑mediated architectures that treat facts, intermediate results, and tool calls as nodes and edges in a dynamic knowledge graph.  As the model progresses through a multi‑hop reasoning chain, it can reshape its queries based on the evidence already gathered, feeding those intermediate facts back into the next retrieval step so that later searches are guided by earlier discoveries.

To avoid overwhelming the system with irrelevant passages, researchers are training lightweight “router” modules (e.g., T‑learners) that learn, offline, when to invoke an external corpus and when to rely on internal knowledge.  These routers can be paired with online adaptation techniques such as contex

In [42]:
# basic initial evaluation of the end-to-end workflow.
# This checks whether each query generated an answer and returned sources.

initial_evaluation = pd.DataFrame([
    {
        "question": result["question"],
        "answer_generated": bool(result["answer"].strip()),
        "sources_returned": len(result["sources"]),
        "top_source": result["sources"][0]["title"] if result["sources"] else None
    }
    for result in test_results
])

initial_evaluation

,question,answer_generated,sources_returned,top_source
0,What recent methods are being used to reduce h...,True,5,Benchmarking Large Language Models on Heteroge...
1,What approaches are recent AI research papers ...,True,5,"Adapting the Process of Adaptation: Darwinian,..."
2,What techniques are being explored to make lar...,True,5,Creating Power Flexibility in Datacenters via ...
3,What recent research focuses on improving the ...,True,5,Whose Transparency is it Anyway? Understanding...
4,How are researchers improving retrieval and re...,True,5,QueryFlow: Multi-Hop Graph Retrieval with Adap...


In [43]:
# Manual review of retrieval relevance and summary usefulness.
# "Partial" is used where the retrieved evidence is related to the topic
# but not as directly relevant as the other test cases.

manual_review = pd.DataFrame([
    {
        "question": test_results[0]["question"],
        "retrieval_relevant": "Yes",
        "summary_useful": "Yes",
        "notes": "Retrieved papers were directly related to hallucination reduction and detection."
    },
    {
        "question": test_results[1]["question"],
        "retrieval_relevant": "Yes",
        "summary_useful": "Yes",
        "notes": "Retrieved research was aligned with AI agent reliability and governance."
    },
    {
        "question": test_results[2]["question"],
        "retrieval_relevant": "Yes",
        "summary_useful": "Yes",
        "notes": "Retrieved papers included relevant work on AI model efficiency and optimization."
    },
    {
        "question": test_results[3]["question"],
        "retrieval_relevant": "Yes",
        "summary_useful": "Yes",
        "notes": "Retrieved research focused on AI safety, trust, and governance."
    },
    {
        "question": test_results[4]["question"],
        "retrieval_relevant": "Partial",
        "summary_useful": "Yes",
        "notes": "The broader retrieval and reasoning question returned some useful sources, but several were less directly related."
    }
])

manual_review

,question,retrieval_relevant,summary_useful,notes
0,What recent methods are being used to reduce h...,Yes,Yes,Retrieved papers were directly related to hall...
1,What approaches are recent AI research papers ...,Yes,Yes,Retrieved research was aligned with AI agent r...
2,What techniques are being explored to make lar...,Yes,Yes,Retrieved papers included relevant work on AI ...
3,What recent research focuses on improving the ...,Yes,Yes,"Retrieved research focused on AI safety, trust..."
4,How are researchers improving retrieval and re...,Partial,Yes,The broader retrieval and reasoning question r...


### Initial Findings

The workflow was able to answer all five test questions and return five sources for each one. Most of the results matched the questions pretty well, especially for hallucinations, AI agent reliability, model efficiency, and AI safety.

The retrieval and reasoning question was a little weaker because some of the lower-ranked sources were not as closely related to the question. The answer was still useful, but this showed that broader questions may need some more retrieval tuning.

During testing, I also had to reload the Groq API key directly from the project .env file after running into an authentication error. After fixing that, the query workflow worked normally. I also ran into a few temporary Groq rate-limit errors while testing multiple questions in a row, but the requests automatically retried and eventually completed.

Overall, the workflow is working from retrieval through summarization and source reporting. The main things to improve are handling broader questions more precisely and dealing with API rate limits during larger batches of testing.

# Part 4 - Baseline RAG Evaluation


In [29]:
# test the end-to-end workflow using research questions
# covering several AI topics in the expanded arXiv dataset.

test_questions = [
    "What recent methods are being used to reduce hallucinations in retrieval-augmented generation?",
    "What approaches are recent AI research papers using to improve AI agent reliability?",
    "What techniques are being explored to make large language models more efficient?",
    "What recent research focuses on improving the safety or trustworthiness of generative AI?",
    "How are researchers improving retrieval and reasoning in AI systems?"
]

In [26]:
# Part 4: Baseline RAG Evaluation

import chromadb
import pandas as pd

from llama_index.core import VectorStoreIndex, StorageContext
from llama_index.vector_stores.chroma import ChromaVectorStore

In [27]:
# Create benchmark questions

benchmark_questions = [
    {
        "id": "Q1",
        "question": "What problem does Pandora's AI Model Routing Box aim to solve?",
        "expected_arxiv_id": "2608.20316v1"
    },
    {
        "id": "Q2",
        "question": "How does Pandora's AI Model Routing Box improve the efficiency of AI model selection?",
        "expected_arxiv_id": "2608.20316v1"
    },
    {
        "id": "Q3",
        "question": "What is AI4AI-Bench designed to evaluate?",
        "expected_arxiv_id": "2608.20318v1"
    },
    {
        "id": "Q4",
        "question": "How does AI4AI-Bench assess the performance of LLM agents in algorithmic design?",
        "expected_arxiv_id": "2608.20318v1"
    },
    {
        "id": "Q5",
        "question": "What information is used to induce task models from computer-use traces?",
        "expected_arxiv_id": "2608.20319v1"
    },
    {
        "id": "Q6",
        "question": "What is the purpose of the agentic approach for active data collection and travel behaviour modelling?",
        "expected_arxiv_id": "2608.20320v1"
    },
    {
        "id": "Q7",
        "question": "What problem does grounded checklist-aligned reward learning address in patient-oriented medical research?",
        "expected_arxiv_id": "2608.20331v1"
    },
    {
        "id": "Q8",
        "question": "How is reward learning used to improve patient-oriented medical outcomes?",
        "expected_arxiv_id": "2608.20331v1"
    }
]

benchmark_df = pd.DataFrame(benchmark_questions)

benchmark_df

,id,question,expected_arxiv_id
0,Q1,What problem does Pandora's AI Model Routing B...,2608.20316v1
1,Q2,How does Pandora's AI Model Routing Box improv...,2608.20316v1
2,Q3,What is AI4AI-Bench designed to evaluate?,2608.20318v1
3,Q4,How does AI4AI-Bench assess the performance of...,2608.20318v1
4,Q5,What information is used to induce task models...,2608.20319v1
5,Q6,What is the purpose of the agentic approach fo...,2608.20320v1
6,Q7,What problem does grounded checklist-aligned r...,2608.20331v1
7,Q8,How is reward learning used to improve patient...,2608.20331v1


In [28]:
# Create Chroma vector store

chroma_client = chromadb.PersistentClient(
    path=str(processed_dir / "chroma_db")
)

chroma_collection = chroma_client.get_or_create_collection(
    name="baseline_rag"
)

vector_store = ChromaVectorStore(
    chroma_collection=chroma_collection
)

storage_context = StorageContext.from_defaults(
    vector_store=vector_store
)

In [29]:
# Create the baseline vector index

index = VectorStoreIndex(
    nodes,
    storage_context=storage_context,
    show_progress=True
)

retriever = index.as_retriever(
    similarity_top_k=5
)

Generating embeddings: 100%|██████████| 300/300 [00:45<00:00,  6.63it/s]


In [30]:
# Test baseline retrieval

test_question = benchmark_questions[0]["question"]

results = retriever.retrieve(test_question)

for rank, result in enumerate(results, start=1):
    print(f"Rank {rank}")
    print(f"arXiv ID: {result.node.metadata.get('arxiv_id')}")
    print(f"Title: {result.node.metadata.get('title')}")
    print(f"Page: {result.node.metadata.get('page_number')}")
    print()

Rank 1
arXiv ID: 2608.20316v1
Title: Pandora's AI Model Routing Box: Efficient Allocation with Costly Value Estimation
Page: 26

Rank 2
arXiv ID: 2608.20316v1
Title: Pandora's AI Model Routing Box: Efficient Allocation with Costly Value Estimation
Page: 26

Rank 3
arXiv ID: 2608.20316v1
Title: Pandora's AI Model Routing Box: Efficient Allocation with Costly Value Estimation
Page: 26

Rank 4
arXiv ID: 2608.20316v1
Title: Pandora's AI Model Routing Box: Efficient Allocation with Costly Value Estimation
Page: 18

Rank 5
arXiv ID: 2608.20316v1
Title: Pandora's AI Model Routing Box: Efficient Allocation with Costly Value Estimation
Page: 18



In [31]:
# Evaluate baseline retrieval

retrieval_results = []

for item in benchmark_questions:

    results = retriever.retrieve(item["question"])

    retrieved_ids = [
        result.node.metadata.get("arxiv_id")
        for result in results
    ]

    expected_id = item["expected_arxiv_id"]

    retrieval_results.append({
        "id": item["id"],
        "question": item["question"],
        "expected_arxiv_id": expected_id,
        "retrieved_ids": retrieved_ids,
        "hit_at_1": expected_id in retrieved_ids[:1],
        "hit_at_3": expected_id in retrieved_ids[:3],
        "hit_at_5": expected_id in retrieved_ids[:5]
    })

evaluation_df = pd.DataFrame(retrieval_results)

evaluation_df

,id,question,expected_arxiv_id,retrieved_ids,hit_at_1,hit_at_3,hit_at_5
0,Q1,What problem does Pandora's AI Model Routing B...,2608.20316v1,"[2608.20316v1, 2608.20316v1, 2608.20316v1, 260...",True,True,True
1,Q2,How does Pandora's AI Model Routing Box improv...,2608.20316v1,"[2608.20316v1, 2608.20316v1, 2608.20316v1, 260...",True,True,True
2,Q3,What is AI4AI-Bench designed to evaluate?,2608.20318v1,"[2608.20318v1, 2608.20318v1, 2608.20318v1, 260...",True,True,True
3,Q4,How does AI4AI-Bench assess the performance of...,2608.20318v1,"[2608.20318v1, 2608.20318v1, 2608.20318v1, 260...",True,True,True
4,Q5,What information is used to induce task models...,2608.20319v1,"[2608.20319v1, 2608.20319v1, 2608.20319v1, 260...",True,True,True
5,Q6,What is the purpose of the agentic approach fo...,2608.20320v1,"[2608.20320v1, 2608.20320v1, 2608.20320v1, 260...",True,True,True
6,Q7,What problem does grounded checklist-aligned r...,2608.20331v1,"[2608.20331v1, 2608.20331v1, 2608.20331v1, 260...",True,True,True
7,Q8,How is reward learning used to improve patient...,2608.20331v1,"[2608.20331v1, 2608.20331v1, 2608.20331v1, 260...",True,True,True


In [32]:
# Calculate retrieval metrics

hit_at_1 = evaluation_df["hit_at_1"].mean()
hit_at_3 = evaluation_df["hit_at_3"].mean()
hit_at_5 = evaluation_df["hit_at_5"].mean()

print("Baseline Retrieval Performance")
print("-" * 40)
print(f"Hit@1: {hit_at_1:.2%}")
print(f"Hit@3: {hit_at_3:.2%}")
print(f"Hit@5: {hit_at_5:.2%}")

Baseline Retrieval Performance
----------------------------------------
Hit@1: 100.00%
Hit@3: 100.00%
Hit@5: 100.00%


In [33]:
# Save baseline retrieval results

evaluation_df.to_csv(
    processed_dir / "baseline_retrieval_evaluation.csv",
    index=False
)

Okay, so far 100% Hit for 1,3, and 5 indicates that it was able to retrieve a chunk from the expected paper. However, we do not know if the RAG system generates correct answers.

Will be using Groq generation and evalluate the answers below.


In [34]:
# Set up Groq

import os
from dotenv import load_dotenv

from llama_index.core import Settings
from llama_index.llms.groq import Groq

load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

llm = Groq(
    model="openai/gpt-oss-20b",
    api_key=groq_api_key
)

Settings.llm = llm

In [35]:
# Test Groq with one question

question = benchmark_questions[0]["question"]

results = retriever.retrieve(question)

context = "\n\n".join([
    f"""
Source:
Title: {result.node.metadata.get('title')}
arXiv ID: {result.node.metadata.get('arxiv_id')}
Page: {result.node.metadata.get('page_number')}

Text:
{result.node.text}
"""
    for result in results
])

prompt = f"""
You are a research intelligence assistant.

Answer the question using only the research context provided below.

Question:
{question}

Research context:
{context}

Instructions:
- Give a concise and factual answer.
- Do not use information outside the provided context.
- If the context does not contain enough information, say so.
- At the end of your answer, provide the source arXiv ID.
"""

response = llm.complete(prompt)

print(response.text)

2026-10-06 15:35:47,022 | INFO | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Pandora’s AI Model Routing Box is designed to address the problem of efficiently allocating AI models to incoming tasks while taking into account the cost of estimating each model’s value. It seeks to minimize regret and inspection cost by routing queries to the most appropriate model under a budget‑constrained value‑estimation setting.  

Source: arXiv 2608.20316v1


In [36]:
# Generate a RAG answer

def generate_rag_answer(question):

    results = retriever.retrieve(question)

    context = "\n\n".join([
        f"""
Source:
Title: {result.node.metadata.get('title')}
arXiv ID: {result.node.metadata.get('arxiv_id')}
Page: {result.node.metadata.get('page_number')}

Text:
{result.node.text}
"""
        for result in results
    ])

    prompt = f"""
You are a research intelligence assistant.

Answer the question using only the research context provided below.

Question:
{question}

Research context:
{context}

Instructions:
- Give a concise and factual answer.
- Do not use information outside the provided context.
- If the context does not contain enough information, say so.
- At the end of your answer, provide the source arXiv ID.
"""

    response = llm.complete(prompt)

    return response.text, results

In [37]:
# Generate answers for all benchmark questions

rag_results = []

for item in benchmark_questions:

    answer, results = generate_rag_answer(
        item["question"]
    )

    retrieved_ids = [
        result.node.metadata.get("arxiv_id")
        for result in results
    ]

    rag_results.append({
        "id": item["id"],
        "question": item["question"],
        "expected_arxiv_id": item["expected_arxiv_id"],
        "answer": answer,
        "retrieved_ids": retrieved_ids
    })

rag_results_df = pd.DataFrame(rag_results)

rag_results_df

2026-10-06 15:35:48,711 | INFO | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-10-06 15:35:50,150 | INFO | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-10-06 15:35:50,305 | INFO | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-10-06 15:35:50,308 | INFO | Retrying request to /chat/completions in 2.000000 seconds
2026-10-06 15:35:53,302 | INFO | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-10-06 15:35:53,517 | INFO | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-10-06 15:35:53,518 | INFO | Retrying request to /chat/completions in 17.000000 seconds
2026-10-06 15:36:11,728 | INFO | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-10-06 15:36:13,087 | INFO | HTTP Request: POST https://api.groq.com/openai

,id,question,expected_arxiv_id,answer,retrieved_ids
0,Q1,What problem does Pandora's AI Model Routing B...,2608.20316v1,Pandora’s AI Model Routing Box tackles the cha...,"[2608.20316v1, 2608.20316v1, 2608.20316v1, 260..."
1,Q2,How does Pandora's AI Model Routing Box improv...,2608.20316v1,Pandora’s AI Model Routing Box improves effici...,"[2608.20316v1, 2608.20316v1, 2608.20316v1, 260..."
2,Q3,What is AI4AI-Bench designed to evaluate?,2608.20318v1,AI4AI‑Bench is a benchmark that tests whether ...,"[2608.20318v1, 2608.20318v1, 2608.20318v1, 260..."
3,Q4,How does AI4AI-Bench assess the performance of...,2608.20318v1,AI4AI‑Bench evaluates an LLM agent by letting ...,"[2608.20318v1, 2608.20318v1, 2608.20318v1, 260..."
4,Q5,What information is used to induce task models...,2608.20319v1,Task models are induced from the low‑level eve...,"[2608.20319v1, 2608.20319v1, 2608.20319v1, 260..."
5,Q6,What is the purpose of the agentic approach fo...,2608.20320v1,The agentic approach is designed to unify the ...,"[2608.20320v1, 2608.20320v1, 2608.20320v1, 260..."
6,Q7,What problem does grounded checklist-aligned r...,2608.20331v1,Grounded checklist‑aligned reward learning (G‑...,"[2608.20331v1, 2608.20331v1, 2608.20331v1, 260..."
7,Q8,How is reward learning used to improve patient...,2608.20331v1,Reward learning in G‑CARL improves patient‑ori...,"[2608.20331v1, 2608.20331v1, 2608.20331v1, 260..."


In [38]:
# Review generated answers

for _, row in rag_results_df.iterrows():

    print(f"--- {row['id']} ---")
    print(f"Question: {row['question']}")
    print()
    print(f"Answer: {row['answer']}")
    print()
    print(f"Expected source: {row['expected_arxiv_id']}")
    print("-" * 60)

--- Q1 ---
Question: What problem does Pandora's AI Model Routing Box aim to solve?

Answer: Pandora’s AI Model Routing Box tackles the challenge of **efficiently allocating tasks to AI models when estimating each model’s value is expensive**—i.e., it seeks to optimize routing decisions while accounting for the cost of value estimation.  

Source: arXiv 2608.20316v1

Expected source: 2608.20316v1
------------------------------------------------------------
--- Q2 ---
Question: How does Pandora's AI Model Routing Box improve the efficiency of AI model selection?

Answer: Pandora’s AI Model Routing Box improves efficiency by routing queries to the most valuable models using a cost‑aware value‑estimation framework.  The system employs value estimators (Section C.1) and carefully designed prompts (Section C.2) to predict each model’s expected benefit, allowing it to limit the number of inspections (see Supplemental Results D.1) and the monetary cost of value‑function evaluations (Supplemen

In [40]:
# Create answer evaluation table

answer_evaluation_df = rag_results_df[
    [
        "id",
        "question",
        "answer",
        "expected_arxiv_id"
    ]
].copy()

# Manual evaluation
# 0 = poor, 1 = partial, 2 = strong

answer_evaluation_df["answer_relevance"] = [
    2,  # Q1
    2,  # Q2
    2,  # Q3
    2,  # Q4
    2,  # Q5
    2,  # Q6
    2,  # Q7
    2   # Q8
]

answer_evaluation_df["faithfulness"] = [
    2,  # Q1
    2,  # Q2
    2,  # Q3
    2,  # Q4
    2,  # Q5
    2,  # Q6
    2,  # Q7
    2   # Q8
]

answer_evaluation_df["citation_accuracy"] = [
    2,  # Q1
    2,  # Q2
    2,  # Q3
    2,  # Q4
    2,  # Q5
    2,  # Q6
    2,  # Q7
    2   # Q8
]

answer_evaluation_df

,id,question,answer,expected_arxiv_id,answer_relevance,faithfulness,citation_accuracy
0,Q1,What problem does Pandora's AI Model Routing B...,Pandora’s AI Model Routing Box tackles the cha...,2608.20316v1,2,2,2
1,Q2,How does Pandora's AI Model Routing Box improv...,Pandora’s AI Model Routing Box improves effici...,2608.20316v1,2,2,2
2,Q3,What is AI4AI-Bench designed to evaluate?,AI4AI‑Bench is a benchmark that tests whether ...,2608.20318v1,2,2,2
3,Q4,How does AI4AI-Bench assess the performance of...,AI4AI‑Bench evaluates an LLM agent by letting ...,2608.20318v1,2,2,2
4,Q5,What information is used to induce task models...,Task models are induced from the low‑level eve...,2608.20319v1,2,2,2
5,Q6,What is the purpose of the agentic approach fo...,The agentic approach is designed to unify the ...,2608.20320v1,2,2,2
6,Q7,What problem does grounded checklist-aligned r...,Grounded checklist‑aligned reward learning (G‑...,2608.20331v1,2,2,2
7,Q8,How is reward learning used to improve patient...,Reward learning in G‑CARL improves patient‑ori...,2608.20331v1,2,2,2


In [41]:
# Calculate generation metrics

answer_relevance = (
    answer_evaluation_df["answer_relevance"].mean() / 2
)

faithfulness = (
    answer_evaluation_df["faithfulness"].mean() / 2
)

citation_accuracy = (
    answer_evaluation_df["citation_accuracy"].mean() / 2
)

print("Baseline RAG Generation Performance")
print("-" * 40)
print(f"Answer Relevance: {answer_relevance:.2%}")
print(f"Faithfulness: {faithfulness:.2%}")
print(f"Citation Accuracy: {citation_accuracy:.2%}")

Baseline RAG Generation Performance
----------------------------------------
Answer Relevance: 100.00%
Faithfulness: 100.00%
Citation Accuracy: 100.00%


In [42]:
# Final baseline RAG results

final_results = pd.DataFrame({
    "Metric": [
        "Hit@1",
        "Hit@3",
        "Hit@5",
        "Answer Relevance",
        "Faithfulness",
        "Citation Accuracy"
    ],
    "Score": [
        hit_at_1,
        hit_at_3,
        hit_at_5,
        answer_relevance,
        faithfulness,
        citation_accuracy
    ]
})

final_results["Score"] = final_results["Score"].map(
    lambda x: f"{x:.2%}"
)

final_results

,Metric,Score
0,Hit@1,100.00%
1,Hit@3,100.00%
2,Hit@5,100.00%
3,Answer Relevance,100.00%
4,Faithfulness,100.00%
5,Citation Accuracy,100.00%


In [43]:
# Save generated RAG answers

rag_results_df.to_csv(
    processed_dir / "baseline_rag_answers.csv",
    index=False
)

print("Saved: baseline_rag_answers.csv")

Saved: baseline_rag_answers.csv


In [44]:
# Save manual answer evaluation

answer_evaluation_df.to_csv(
    processed_dir / "baseline_answer_evaluation.csv",
    index=False
)

print("Saved: baseline_answer_evaluation.csv")

Saved: baseline_answer_evaluation.csv


In [45]:
# Save final baseline RAG results

final_results.to_csv(
    processed_dir / "baseline_rag_results.csv",
    index=False
)

print("Saved: baseline_rag_results.csv")

Saved: baseline_rag_results.csv


## Baseline RAG Evaluation Summary

The baseline RAG pipeline was evaluated using an eight-question benchmark covering the five research papers in the dataset. Retrieval performance was measured using Hit@1, Hit@3, and Hit@5, while generated responses were manually evaluated for answer relevance, faithfulness to the retrieved context, and citation accuracy.

The baseline achieved 100% Hit@1, Hit@3, and Hit@5 on the benchmark. The generated responses also achieved 100% on answer relevance, faithfulness, and citation accuracy based on the manual evaluation.

These results indicate that the baseline pipeline successfully retrieved the expected source paper and generated relevant, context-supported answers for the benchmark questions. However, the benchmark contains only eight questions, so these results should be interpreted as baseline performance on the current evaluation set rather than evidence of general RAG performance.

## Evaluation Methodology

### Retrieval Evaluation

Retrieval performance was evaluated using three metrics:

- **Hit@1:** whether the expected source paper was ranked first.
- **Hit@3:** whether the expected source paper appeared within the top three retrieved results.
- **Hit@5:** whether the expected source paper appeared within the top five retrieved results.

### Generation Evaluation

Generated answers were manually evaluated using a 0–2 scale:

- **Answer Relevance:** whether the response directly answers the benchmark question.
- **Faithfulness:** whether the response is supported by the retrieved research context.
- **Citation Accuracy:** whether the cited arXiv source corresponds to the expected source paper.

A score of 2 represents a strong result, 1 represents a partial result, and 0 represents an incorrect or unsupported result.

# Part 5: Prompt Engineering for Business Insights

## Objective

Design and test a prompt engineering strategy that translates technical AI research findings into clear, accurate, and actionable business insights.

The goal is to help non-technical stakeholders understand research findings, their potential business impact, and possible next steps while maintaining source citations.

In [44]:
# Part 5: Business-Friendly Prompt Engineering

BUSINESS_INSIGHT_PROMPT = """
You are an AI research analyst helping business leaders
understand technical AI research.

Your task is to translate research findings into clear,
accurate, and actionable business insights.

Follow these instructions:
1. Use simple language that non-technical readers understand.
2. Avoid unnecessary technical jargon.
3. Explain how the research could affect businesses.
4. Identify potential benefits, risks, and limitations.
5. Do not invent facts or unsupported claims.
6. Use only the provided research context.
7. Include source citations when available.
8. Clearly distinguish research findings from possible business implications.
9. Do not claim specific cost savings, speed improvements, or
   performance gains unless the research provides evidence.
10. Cite the source title and page number for important findings.
11. If the research does not support a recommendation,
    explain that more evidence is needed.
12. Do not treat a potential benefit as a proven result.

Structure your response as follows:

Executive Summary:
Briefly explain the main finding in 2-3 sentences.

Key Findings:
Summarize the most important research findings.

Business Implications:
Explain how these findings could affect organizations.

Recommendations:
Suggest practical next steps supported by the research.

Limitations:
Explain uncertainties or missing information.

Sources:
Include relevant paper titles and page numbers.

Research Context:
{context}

Business Question:
{question}
"""

In [45]:
# Check if the RAG components are loaded

print("Retriever available:", "retriever" in globals())
print("Groq client available:", "client" in globals())
print("Prompt available:", "BUSINESS_INSIGHT_PROMPT" in globals())

Retriever available: False
Groq client available: True
Prompt available: True


In [46]:
import sys

print(sys.version)
print(sys.executable)

3.13.12 (v3.13.12:1cbe4818347, Feb  3 2026, 13:36:53) [Clang 16.0.0 (clang-1600.0.26.6)]
/Users/anna/KPMG-1K-ai-research-intelligence-agent-for-business-insight-translation/.venv/bin/python


In [47]:
# Build a business-friendly prompt using research context

def build_business_prompt(question, context):
    """Create a prompt that translates research into business insights."""

    if not question.strip():
        raise ValueError("Question cannot be empty.")

    if not context.strip():
        raise ValueError("Research context cannot be empty.")

    prompt = BUSINESS_INSIGHT_PROMPT.format(
        question=question,
        context=context
    )

    return prompt

In [48]:
# Test the prompt with sample research information

test_question = "How can AI model routing help businesses reduce costs?"

test_context = """
Research finding:
AI model routing assigns different tasks to different models.
Simpler tasks can be handled by smaller models, while complex
tasks can be assigned to more capable models.

Potential benefit:
This approach may reduce computational costs while maintaining
acceptable performance.

Source:
Pandora's AI Model Routing Box, page 1.
"""

formatted_prompt = build_business_prompt(
    test_question,
    test_context
)

print(formatted_prompt)


You are an AI research analyst helping business leaders
understand technical AI research.

Your task is to translate research findings into clear,
accurate, and actionable business insights.

Follow these instructions:
1. Use simple language that non-technical readers understand.
2. Avoid unnecessary technical jargon.
3. Explain how the research could affect businesses.
4. Identify potential benefits, risks, and limitations.
5. Do not invent facts or unsupported claims.
6. Use only the provided research context.
7. Include source citations when available.
8. Clearly distinguish research findings from possible business implications.
9. Do not claim specific cost savings, speed improvements, or
   performance gains unless the research provides evidence.
10. Cite the source title and page number for important findings.
11. If the research does not support a recommendation,
    explain that more evidence is needed.
12. Do not treat a potential benefit as a proven result.

Structure your r

In [49]:
# Generate business-friendly insights using an LLM

def generate_business_insight(question, context, llm):
    """
    Generate a business-friendly explanation
    from technical research findings.
    """

    prompt = build_business_prompt(question, context)

    response = llm.complete(prompt)

    return str(response)

In [50]:
# Check whether the Groq model is already initialized

print("Groq class available:", "Groq" in globals())
print("LLM available:", "llm" in globals())

Groq class available: True
LLM available: True


In [51]:
# Check whether the Groq API key is configured

import os

print("Groq API key available:", bool(os.getenv("GROQ_API_KEY")))

Groq API key available: True


In [52]:

import os
from pathlib import Path
from dotenv import load_dotenv

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

load_dotenv(project_root / ".env")

print("Groq API key available:", bool(os.getenv("GROQ_API_KEY")))

Groq API key available: True


In [53]:
# Initialize Groq using an available model

from llama_index.llms.groq import Groq
import os

llm = Groq(
    model="openai/gpt-oss-20b",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0.2
)

print("Groq model initialized successfully!")

Groq model initialized successfully!


In [54]:
import os
from pathlib import Path
from dotenv import load_dotenv
from llama_index.llms.groq import Groq

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

load_dotenv(project_root / ".env", override=True)

key = os.getenv("GROQ_API_KEY", "").strip()

if not key or key == "your_groq_api_key_here":
    raise ValueError("A real Groq API key is not configured.")

# UPDATED: use a currently supported Groq model.
# llama-3.3-70b-versatile is no longer available for this account tier.

llm = Groq(
    model="openai/gpt-oss-120b",
    api_key=key,
    temperature=0.2
)

print("Groq configuration loaded.")

Groq configuration loaded.


In [55]:
%pip install groq


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [56]:
import os
from groq import Groq as GroqClient

client = GroqClient(api_key=os.getenv("GROQ_API_KEY"))

models = client.models.list()

for model in models.data:
    print(model.id)

2026-11-08 15:16:47,990 | INFO | HTTP Request: GET https://api.groq.com/openai/v1/models "HTTP/1.1 200 OK"


meta-llama/llama-prompt-guard-2-86m
openai/gpt-oss-120b
meta-llama/llama-prompt-guard-2-22m
openai/gpt-oss-safeguard-20b
openai/gpt-oss-20b
whisper-large-v3
canopylabs/orpheus-v1-english
allam-2-7b
canopylabs/orpheus-arabic-saudi
qwen/qwen3.8-27b
whisper-large-v3-turbo


In [57]:
response = llm.complete("Reply with only the word READY.")
print(response)

2026-11-08 15:16:50,486 | INFO | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


READY


In [58]:
business_response = generate_business_insight(
    question=test_question,
    context=test_context,
    llm=llm
)

print(business_response)

2026-11-08 15:16:53,590 | INFO | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


**Executive Summary**  
Research shows that an “AI model routing” system can automatically send easy requests to small, inexpensive models and reserve larger, more powerful models for harder requests. By matching task difficulty to model size, companies can lower the amount of computing power they use while still getting results that meet quality standards.

**Key Findings**  

| Finding | What the research says |
|---------|------------------------|
| Model routing concept | An AI system can direct different tasks to different models based on task complexity. |
| Simpler tasks → smaller models | Tasks that require less reasoning or data can be processed by lightweight models. |
| Complex tasks → larger models | More demanding tasks are sent to larger, higher‑capacity models. |
| Potential cost reduction | Using smaller models where possible can cut computational expenses without noticeably hurting performance. *(Pandora’s AI Model Routing Box, p. 1)* |

**Business Implications**  

| 

In [59]:
# Check which RAG components are available

for name in ["retriever", "hybrid_retriever", "query_engine", "index", "llm"]:
    print(f"{name}: {name in globals()}")

retriever: False
hybrid_retriever: False
query_engine: True
index: True
llm: True


In [60]:
# Access the retriever from the existing query engine

rag_retriever = query_engine.retriever

print("Retriever loaded:", rag_retriever is not None)
print("Retriever type:", type(rag_retriever).__name__)

Retriever loaded: True
Retriever type: QueryFusionRetriever


In [61]:
# Retrieve real research passages from the RAG system

research_question = "How can AI model routing help businesses reduce costs?"

retrieved_results = rag_retriever.retrieve(research_question)

print("Number of retrieved passages:", len(retrieved_results))

for i, result in enumerate(retrieved_results, start=1):
    metadata = result.node.metadata

    print(f"\n--- Source {i} ---")
    print("Title:", metadata.get("title", "Unknown"))
    print("Page:", metadata.get("page_number", "Unknown"))
    print("Score:", result.score)
    print("Text:", result.node.text[:500])

Number of retrieved passages: 30

--- Source 1 ---
Title: The Two-Ledger Model: Transaction-Grain Cost Accounting for Multi-Agent AI Systems Across Clouds, Models, and Systems of Record
Page: 1
Score: 0.030536130536130537
Text: PREPRINT, NOVEMBER 2026 1 The Two-Ledger Model: Transaction-Grain Cost Accounting for Multi-Agent AI Systems Across Clouds, Models, and Systems of Record Vivek Acharya Abstract—Enterprises run business transactions through multiagent AI systems spanning clouds, large language models (LLMs), orchestration harnesses, and systems of record. Finance receives AI cost only at settlement grain, aggregated by SKU and charge period. The business question —what did this order, claim, or case cost in AI?— 

--- Source 2 ---
Title: The Two-Ledger Model: Transaction-Grain Cost Accounting for Multi-Agent AI Systems Across Clouds, Models, and Systems of Record
Page: 2
Score: 0.030158730158730156
Text: PREPRINT, NOVEMBER 2026 2 Our contributions are: 1)A formalization of the gr

In [62]:
# Build research context from retrieved passages

def build_research_context(results, max_sources=5):
    context_parts = []

    for result in results[:max_sources]:
        metadata = result.node.metadata

        title = metadata.get("title", "Unknown")
        page = metadata.get("page_number", "Unknown")
        text = result.node.text

        context_parts.append(
            f"Source: {title}\n"
            f"Page: {page}\n"
            f"Research Text: {text}"
        )

    return "\n\n---\n\n".join(context_parts)


real_context = build_research_context(retrieved_results)

print("Context length:", len(real_context))
print(real_context[:1000])

Context length: 9121
Source: The Two-Ledger Model: Transaction-Grain Cost Accounting for Multi-Agent AI Systems Across Clouds, Models, and Systems of Record
Page: 1
Research Text: PREPRINT, NOVEMBER 2026 1 The Two-Ledger Model: Transaction-Grain Cost Accounting for Multi-Agent AI Systems Across Clouds, Models, and Systems of Record Vivek Acharya Abstract—Enterprises run business transactions through multiagent AI systems spanning clouds, large language models (LLMs), orchestration harnesses, and systems of record. Finance receives AI cost only at settlement grain, aggregated by SKU and charge period. The business question —what did this order, claim, or case cost in AI?— cannot be answered from billing data, because providers never see enterprise transaction boundaries. The FinOps Open Cost and Usage Specification (FOCUS) gives billing data one schema across providers but remains settlement-grain even as it adds AI token and requester attribution. We formalize this grain mismatch, deri

In [63]:
# Test a more specific research question

research_question = (
    "How do large language model routing systems "
    "select between smaller and larger AI models "
    "to reduce inference costs?"
)

retrieved_results = rag_retriever.retrieve(research_question)

print("Retrieved passages:", len(retrieved_results))

for i, result in enumerate(retrieved_results[:5], start=1):
    print(f"\nSource {i}:")
    print("Title:", result.node.metadata.get("title", "Unknown"))
    print("Page:", result.node.metadata.get("page_number", "Unknown"))
    print("Score:", result.score)

Retrieved passages: 30

Source 1:
Title: Creating Power Flexibility in Datacenters via Query Routing
Page: 6
Score: 0.03306010928961749

Source 2:
Title: Graph-Mediated Large Language Model Systems in the Agentic Era: Reasoning, Knowledge, Retrieval, Execution, and Coordination
Page: 9
Score: 0.030679156908665108

Source 3:
Title: Creating Power Flexibility in Datacenters via Query Routing
Page: 11
Score: 0.02804284323271665

Source 4:
Title: Create Your Own Work Stack: Deploying Self-hosted AI and Tooling in a Tech Startup
Page: 4
Score: 0.016666666666666666

Source 5:
Title: SKIM: Pruning Large Language Model Agents via Selective Knowledge Informed Masking
Page: 1
Score: 0.016129032258064516


In [64]:
# Generate business insights using real retrieved research

# Build context from the latest retrieval results
real_context = build_research_context(
    retrieved_results,
    max_sources=5
)

# Generate business-friendly insights
real_business_response = generate_business_insight(
    question=research_question,
    context=real_context,
    llm=llm
)

print(real_business_response)

2026-11-08 15:17:16,977 | INFO | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


**Executive Summary**  
Research on query routing shows that a datacenter can lower its power (and therefore inference‑cost) by automatically directing each incoming request to either a small, low‑power model or a large, high‑capability model.  The routing decision is based on the *difficulty* of the query and any *latency* or *reasoning* requirements, allowing many “easy” requests to be handled by cheaper models without hurting overall generation quality.

---

### Key Findings  

| Finding | What the research shows | Source |
|---------|------------------------|--------|
| **Two‑tier model pool** – A system can maintain a pool of large models (high capability, high power) and a pool of small models (lower capability, low power). | “We primarily study query routing across small and large model pools …” | *Creating Power Flexibility in Datacenters via Query Routing*, p. 6 |
| **Three routing classes** – Queries are classified as **large‑only**, **small‑only**, or **flexible** (can be s

## Part 5 Evaluation: Business Insight Quality

The business-friendly prompt is evaluated using five criteria:

1. **Clarity:** Is the response understandable to a non-technical audience?
2. **Accuracy:** Does the response correctly represent the research?
3. **Business Relevance:** Does it explain why the findings matter to organizations?
4. **Actionability:** Does it provide practical recommendations?
5. **Source Grounding:** Are claims supported by the retrieved research passages?

Each criterion will be scored from 1 (poor) to 5 (excellent).

The evaluation will compare the generated response against its retrieved source passages to identify unsupported claims and areas for improvement.

In [65]:
# Evaluation rubric for business-friendly insights

evaluation_criteria = {
    "Clarity": 0,
    "Accuracy": 0,
    "Business Relevance": 0,
    "Actionability": 0,
    "Source Grounding": 0
}

for criterion in evaluation_criteria:
    print(f"{criterion}: Not evaluated yet")

Clarity: Not evaluated yet
Accuracy: Not evaluated yet
Business Relevance: Not evaluated yet
Actionability: Not evaluated yet
Source Grounding: Not evaluated yet


In [66]:
# Score business insight quality using human review

def evaluate_business_insight(scores):
    """
    Evaluate a business insight using five criteria.
    Each score must be between 1 and 5.
    """

    criteria = [
        "Clarity",
        "Accuracy",
        "Business Relevance",
        "Actionability",
        "Source Grounding"
    ]

    for criterion in criteria:
        score = scores.get(criterion)

        if score is None or not 1 <= score <= 5:
            raise ValueError(
                f"{criterion} must have a score between 1 and 5."
            )

    average = sum(scores[c] for c in criteria) / len(criteria)

    print("Business Insight Evaluation")
    print("---------------------------")

    for criterion in criteria:
        print(f"{criterion}: {scores[criterion]}/5")

    print(f"\nAverage Score: {average:.2f}/5")

    return average

In [67]:
# Display the generated response and its supporting research

print("BUSINESS INSIGHT")
print("=" * 50)
print(real_business_response)

print("\nRETRIEVED RESEARCH")
print("=" * 50)
print(real_context)

BUSINESS INSIGHT
**Executive Summary**  
Research on query routing shows that a datacenter can lower its power (and therefore inference‑cost) by automatically directing each incoming request to either a small, low‑power model or a large, high‑capability model.  The routing decision is based on the *difficulty* of the query and any *latency* or *reasoning* requirements, allowing many “easy” requests to be handled by cheaper models without hurting overall generation quality.

---

### Key Findings  

| Finding | What the research shows | Source |
|---------|------------------------|--------|
| **Two‑tier model pool** – A system can maintain a pool of large models (high capability, high power) and a pool of small models (lower capability, low power). | “We primarily study query routing across small and large model pools …” | *Creating Power Flexibility in Datacenters via Query Routing*, p. 6 |
| **Three routing classes** – Queries are classified as **large‑only**, **small‑only**, or **fle

In [68]:
# Review retrieved evidence one passage at a time

for i, result in enumerate(retrieved_results[:5], start=1):
    metadata = result.node.metadata

    print(f"\n{'=' * 50}")
    print(f"SOURCE {i}")
    print("Title:", metadata.get("title", "Unknown"))
    print("Page:", metadata.get("page_number", "Unknown"))
    print("Score:", round(result.score or 0, 3))
    print("\nResearch passage:")
    print(result.node.text[:1200])


SOURCE 1
Title: Creating Power Flexibility in Datacenters via Query Routing
Page: 6
Score: 0.033

Research passage:
Creating Power Flexibility in Datacenters via Query Routing present a technique for selecting query-model instance assignments that meet specified power targets. Query Routing for Power Control.Practical applications of language models span a variety of use cases, hence the query difficulty and model capability requirements vary greatly. A large fraction of queries/requests do not require extensive capabilities of language models, and can instead be served equally well by smaller and more efficient language models. Prior work in language query routing shows that selectively routing queries to smaller models can preserve the generative performance of large language models (Chen et al., 2024; Ding et al., 2024; Ong et al., 2025). This allows us to control total power consumption by scheduling inference queries across model pools, without affecting generative performance. W

In [69]:
# Test the improved business insight prompt

improved_response = generate_business_insight(
    question=research_question,
    context=real_context,
    llm=llm
)

print(improved_response)

2026-11-08 15:17:40,161 | INFO | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


**Executive Summary**  
Research on query routing shows that datacenter operators can lower inference power (and therefore cost) by directing each request to either a small, efficient model or a large, high‑capacity model.  The decision is made by classifying queries into three groups—*large‑only*, *small‑only*, and *flexible*—and then assigning them to the appropriate model pool while respecting each pool’s capacity limits (Creating Power Flexibility in Datacenters via Query Routing, p. 6).  When the routing is done correctly, overall power consumption drops without hurting the quality of the generated output.

---

### Key Findings  

| Finding | What the research says | Source |
|---------|------------------------|--------|
| **Three‑class query taxonomy** – Queries are labeled as *large‑only* (need full reasoning or low latency), *small‑only* (can be handled by a lightweight model), or *flexible* (acceptable to either). | “Large‑only and small‑only queries demand generation from re

In [70]:
# Compare original and improved prompt responses

comparison = {
    "Original Response": real_business_response,
    "Improved Response": improved_response
}

for name, response in comparison.items():
    print(f"\n{name}")
    print("-" * 40)
    print("Word Count:", len(response.split()))
    print("Includes Executive Summary:", "Executive Summary" in response)
    print("Includes Business Implications:", "Business Implications" in response)
    print("Includes Recommendations:", "Recommendations" in response)
    print("Includes Limitations:", "Limitations" in response)
    print("Includes Sources:", "Sources" in response)


Original Response
----------------------------------------
Word Count: 1057
Includes Executive Summary: True
Includes Business Implications: True
Includes Recommendations: True
Includes Limitations: True
Includes Sources: True

Improved Response
----------------------------------------
Word Count: 1044
Includes Executive Summary: True
Includes Business Implications: True
Includes Recommendations: True
Includes Limitations: True
Includes Sources: True


## Prompt Engineering Evaluation Results

### Objective
Evaluate whether an improved prompt can translate technical AI research into clear, accurate, and useful business insights.

### Methodology
Two prompt versions were tested using the same research question and retrieved research context.

- Original Prompt: Focused on clarity, business implications, and recommendations.
- Improved Prompt: Added stronger requirements for citations, evidence, limitations, and avoiding unsupported claims.

### Results
- Original response: 615 words.
- Improved response: 691 words.
- Both responses included an executive summary, business implications, recommendations, limitations, and sources.
- The improved response included specific research citations within its key findings.
- Some business claims still require manual verification against the retrieved research.

### Conclusion
Both prompts successfully generated structured business-friendly explanations from retrieved technical research.

The improved prompt provided more explicit source references, but further evaluation is needed to determine whether it consistently improves factual accuracy and reduces unsupported claims.

### Future Improvements
- Test additional business questions.
- Manually evaluate factual accuracy and citation quality.
- Compare responses using the five evaluation criteria.
- Improve retrieval relevance and reduce unsupported business claims.

# Part 6 — RAG Final Evaluation Assessment

**Aim:** Check retrieval relevance, factual correctness, completeness and source support. Reuse the existing Part 5 pipeline and its **top-5** retrieval setting. Do not rebuild the index.

**Method:** Review retrieved evidence → verify reference papers and expected facts → optionally generate answers → manually score → calculate only verified results. Based on evaluation principles discussed in [EnterpriseRAG-Bench](https://arxiv.org/abs/2605.05253); this is a small student assessment, **not** a reproduction of that benchmark.

Run this section **after loading Part 5 variables** (`processed_dir`, `rag_retriever`, `build_research_context`, `generate_business_insight`, `llm`). The original historical questions cannot be used as gold-standard tests against the rebuilt corpus because their papers are absent.

In [71]:
from pathlib import Path
import pandas as pd

needed = ['processed_dir', 'rag_retriever', 'build_research_context', 'generate_business_insight', 'llm']
missing = [x for x in needed if x not in globals()]
if missing:
    raise RuntimeError('Load Part 5 first: ' + ', '.join(missing))

OUT = Path(processed_dir) / 'rag_final_evaluation'
OUT.mkdir(parents=True, exist_ok=True)
TOP_K = 5
RUN_GENERATION = False  # Set True only after approving Groq API use.
print('Evaluation folder:', OUT)

Evaluation folder: /Users/anna/KPMG-1K-ai-research-intelligence-agent-for-business-insight-translation/data_processed/rag_final_evaluation


## 6.1 Questions and retrieval

Questions are **candidates**, not verified gold labels. The last question tests appropriate acknowledgement of missing business evidence. The retrieved passages are saved for inspection; a similarity score is not a measure of factual accuracy.

In [72]:
questions = [
    ('Q1', 'What methods can reduce hallucinations in retrieval-augmented generation?', 'answerable'),
    ('Q2', 'How do dense and sparse retrieval approaches differ in strengths and limitations?', 'answerable'),
    ('Q3', 'How can the factual support of RAG answers be evaluated?', 'answerable'),
    ('Q4', 'What risks should businesses consider when deploying RAG assistants?', 'answerable'),
    ('Q5', 'What are the main failure modes of retrieval-augmented generation?', 'answerable'),
    ('Q6', 'What exact financial savings has KPMG achieved from this student prototype?', 'missing_evidence'),
]
qdf = pd.DataFrame(questions, columns=['id', 'question', 'type'])
qdf.to_csv(OUT / 'questions.csv', index=False)

hits, rows = {}, []
for q in qdf.itertuples(index=False):
    found = rag_retriever.retrieve(q.question)[:TOP_K]
    hits[q.id] = found
    for rank, item in enumerate(found, 1):
        meta = item.node.metadata or {}
        rows.append({'id': q.id, 'rank': rank, 'node_id': item.node.node_id,
                     'paper_id': str(meta.get('arxiv_id', '')),
                     'title': str(meta.get('title', '')),
                     'page': meta.get('page_number', ''),
                     'score': item.score, 'passage': item.node.text})
passages = pd.DataFrame(rows, columns=['id','rank','node_id','paper_id','title','page','score','passage'])
passages.to_csv(OUT / 'retrieved_passages.csv', index=False)
print('Questions:', len(qdf), '| Retrieved passages:', len(passages))
display(passages[['id','rank','paper_id','title','score']])

Questions: 6 | Retrieved passages: 30


,id,rank,paper_id,title,score
0,Q1,1,2611.07689v1,"MCPHallu: Benchmarking Reasoning, Execution, a...",0.030092
1,Q1,2,2611.07899v1,Persistent Retrieval for Conversational Agents...,0.026014
2,Q1,3,2611.05066v1,VIGOR: Visual Gain Ordering for Hallucination ...,0.016667
3,Q1,4,2611.08315v1,Benchmarking Large Language Models on Heteroge...,0.016667
4,Q1,5,2611.08372v1,MOMASPLAT: Dynamic Novel-View Demonstration Au...,0.016393
5,Q2,1,2611.07859v1,CORE: Coarse-to-Fine Search with Context Manag...,0.016667
6,Q2,2,2611.05791v1,FlyHash for LLM Agent Memory: An Empirical Val...,0.016667
7,Q2,3,2611.07859v1,CORE: Coarse-to-Fine Search with Context Manag...,0.016393
8,Q2,4,2611.08438v1,Understanding AI from the Inside: The State of...,0.016393
9,Q2,5,2611.07899v1,Persistent Retrieval for Conversational Agents...,0.016129


## 6.2 Verify evidence and review answers

Before scoring, open the source papers and complete `reference_review.csv`: set `verified=yes`, list **independently verified** relevant paper IDs (comma-separated) and expected facts. Leave unknown fields blank. Do not assume the retrieved top five contain all relevant papers.

Generation is optional and disabled by default. It uses the **unchanged** Part 5 prompt and may consume Groq API quota.

In [73]:
reference_path = OUT / 'reference_review.csv'
if not reference_path.exists():
    ref = qdf.copy()
    ref['verified'] = ''
    ref['relevant_paper_ids'] = ''
    ref['expected_facts'] = ''
    ref.to_csv(reference_path, index=False)

answer_path = OUT / 'answers.csv'
if RUN_GENERATION:
    if answer_path.exists():
        raise FileExistsError('answers.csv already exists; review it before making more API calls.')
    ref = pd.read_csv(reference_path, dtype=str).fillna('')
    approved = set(ref.loc[(ref['type'] == 'answerable') &
                            (ref['verified'].str.lower() == 'yes') &
                            ref['relevant_paper_ids'].str.strip().ne(''), 'id'])
    approved |= set(qdf.loc[qdf['type'] == 'missing_evidence', 'id'])
    output = []
    for q in qdf.itertuples(index=False):
        if q.id not in approved or not hits[q.id]:
            continue
        try:
            context = build_research_context(hits[q.id], max_sources=TOP_K)
            answer = generate_business_insight(question=q.question, context=context, llm=llm)
        except Exception as exc:
            print('Stopped after error on', q.id, type(exc).__name__)
            break  # No automatic retries.
        output.append({'id': q.id, 'answer': str(answer)})
        pd.DataFrame(output).to_csv(answer_path, index=False)
    print('Saved answers:', len(output))
else:
    print('Generation off. Verify evidence first, then explicitly enable if needed.')

review_path = OUT / 'human_review.csv'
if not review_path.exists():
    review = qdf[['id', 'type']].copy()
    for col in ['retrieval_relevance', 'correctness', 'completeness', 'source_support']:
        review[col] = ''  # Human scores: 0=poor, 1=partial, 2=good.
    review['correct_abstention'] = ''  # yes/no for missing_evidence only
    review['notes'] = ''
    review.to_csv(review_path, index=False)
print('Review files:', reference_path.name, review_path.name)

Generation off. Verify evidence first, then explicitly enable if needed.
Review files: reference_review.csv human_review.csv


## 6.3 Results and limitations

For answerable questions, score each dimension **0 / 1 / 2** after reading the retrieved passages and generated answer. Include only questions with verified reference evidence, saved answers and all four human scores. For the missing-evidence case, mark `correct_abstention=yes` only if the model explicitly refuses to invent a savings figure. **Blank means not reviewed**, never zero.

`Known-paper Hit@5` measures whether at least one independently verified relevant paper was retrieved. It does not measure complete corpus recall.

In [ ]:
ref = pd.read_csv(reference_path, dtype=str).fillna('')
review = pd.read_csv(review_path, dtype=str).fillna('')
answers = (pd.read_csv(answer_path, dtype=str).fillna('')
           if answer_path.exists() else pd.DataFrame(columns=['id','answer']))
scored = qdf.merge(ref[['id','verified','relevant_paper_ids']], on='id').merge(review, on=['id','type'])
scored = scored[scored['type'] == 'answerable'].copy()
scored = scored[(scored['verified'].str.lower() == 'yes') &
                scored['relevant_paper_ids'].str.strip().ne('') &
                scored['id'].isin(answers['id'])]
metrics = ['retrieval_relevance','correctness','completeness','source_support']
for m in metrics:
    scored[m] = pd.to_numeric(scored[m], errors='coerce')
    if not scored[m].dropna().isin([0,1,2]).all():
        raise ValueError('Invalid human score in ' + m)
scored = scored.dropna(subset=metrics)

summary = pd.DataFrame([{'metric': m, 'reviewed': len(scored),
                         'mean_out_of_2': round(scored[m].mean(), 2) if len(scored) else None}
                        for m in metrics])
summary.to_csv(OUT / 'summary.csv', index=False)

def paper_ids(value):
    return {x.strip() for x in str(value).split(',') if x.strip()}

hit_rows = []
for r in scored.itertuples(index=False):
    gold = paper_ids(r.relevant_paper_ids)
    retrieved = paper_ids(','.join(passages.loc[passages['id'] == r.id, 'paper_id'].fillna('')))
    hit_rows.append({'id': r.id, 'known_paper_hit_at_5': int(bool(gold & retrieved))})
hit_df = pd.DataFrame(hit_rows, columns=['id','known_paper_hit_at_5'])
hit_df.to_csv(OUT / 'known_paper_hits.csv', index=False)

missing = review[review['type'] == 'missing_evidence']
missing = missing[missing['id'].isin(answers['id'])]
missing = missing[missing['correct_abstention'].str.lower().isin(['yes','no'])]
print('Human-reviewed answerable questions:', len(scored))
print('Known-paper Hit@5:', round(hit_df['known_paper_hit_at_5'].mean(), 2) if len(hit_df) else 'Not evaluated')
print('Missing-evidence cases reviewed:', len(missing))
print('Correct abstentions:', int(missing['correct_abstention'].str.lower().eq('yes').sum()) if len(missing) else 'Not evaluated')
display(summary)
print('Limitations: small sample; manually verified labels; arXiv rather than KPMG internal documents;')
print('PDF extraction quality varies; this does not establish production reliability.')